# Document type: DIVA-UTP dataset

The DIVA-UTP dataset (1,800 pages from six books of the *Utopia, Armarium codicum bibliophilorum*
collection on e-codices, CC BY-NC 4.0) has **no document-type labels**, only pixel masks
(background, decoration, filler, text, main body, drop caps). All pages come from bound books,
so they cover two of our types: **codex** and **illuminated**.

This notebook:

1. **Weak labels:** turns each mask into a label: how much of the page is decoration?
2. **Baseline:** runs `ufal/vit-historical-page` unchanged on the DIVA test split.
3. **First fine-tune:** trains the same model on *codex vs illuminated* (train split).
4. **The real check:** tests that model on our own hand-picked codex and illuminated pages,
   which come from other sources.

Before you start:

1. On the Mac, right-click the `Dataset` folder → **Compress**. Upload `Dataset.zip` to **My Drive**.
2. (For step 4) the `pages` folder with `labels.csv` should be in My Drive, or as a shortcut there.
3. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

In [ ]:
!pip install -q -U transformers accelerate
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pathlib, zipfile

ZIP = pathlib.Path('/content/drive/MyDrive/Dataset.zip')      # change if it is elsewhere
PAGES = pathlib.Path('/content/drive/MyDrive/pages')           # our 54 test pages (step 4)

DATA = pathlib.Path('/content/data')
if not DATA.exists():
    zipfile.ZipFile(ZIP).extractall(DATA)                      # local copy: much faster than Drive
# The zip may contain a top folder ("Dataset/"); find the folder that has train/ val/ test/.
ROOT = next(p.parent for p in DATA.rglob('train') if (p.parent / 'test').exists())
for split in ['train', 'val', 'test']:
    print(split, len(list((ROOT / split / 'images').glob('*.jpg'))), 'images')

## 1. Weak labels from the masks

For each page: **decoration share** = (decoration + drop-cap pixels) / (all pixels that are not background).

| Decoration share | Label |
|---|---|
| below 0.15 | `codex` (text pages, small initials) |
| 0.15 to 0.35 | `unsure` (e.g. a border on one side), left out of training |
| 0.35 and more | `illuminated` (borders around the text, miniatures) |

The thresholds come from looking at 24 pages. Check the grid below and move them if needed.

In [ ]:
import numpy as np, pandas as pd
from PIL import Image

CODEX_MAX, ILLUM_MIN = 0.15, 0.35
DECOR, DROPCAP = (255, 0, 0), (0, 255, 255)

def decoration_share(gt_path):
    a = np.asarray(Image.open(gt_path).convert('RGB'))
    background = (a == 0).all(-1)
    deco = (a == DECOR).all(-1) | (a == DROPCAP).all(-1)
    return deco.sum() / max((~background).sum(), 1)

def weak_label(share):
    if share < CODEX_MAX:
        return 'codex'
    if share >= ILLUM_MIN:
        return 'illuminated'
    return 'unsure'

rows = []
for split in ['train', 'val', 'test']:
    for img in sorted((ROOT / split / 'images').glob('*.jpg')):
        gt = ROOT / split / 'GT' / (img.stem + '.gif')
        share = decoration_share(gt)
        rows.append({'split': split, 'file': img.name, 'book': img.name.split('_')[0],
                     'decoration_share': round(float(share), 3), 'label': weak_label(share)})
diva = pd.DataFrame(rows)
diva.to_csv('/content/drive/MyDrive/diva_weak_labels.csv', index=False)
display(pd.crosstab(diva['split'], diva['label']))
display(pd.crosstab(diva['book'], diva['label']))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(diva['decoration_share'], bins=50, color='#1f4e79')
for x in (CODEX_MAX, ILLUM_MIN):
    ax.axvline(x, color='#b23a3a', ls='--')
ax.set_xlabel('decoration share'); ax.set_ylabel('pages'); ax.set_title('DIVA-UTP: how decorated are the pages?')
plt.tight_layout(); plt.show()

# 6 random pages per label, to check the thresholds by eye
fig, axes = plt.subplots(3, 6, figsize=(15, 9))
for row, lab in zip(axes, ['codex', 'unsure', 'illuminated']):
    sample = diva[diva.label == lab].sample(min(6, (diva.label == lab).sum()), random_state=0)
    for ax, (_, r) in zip(row, sample.iterrows()):
        ax.imshow(Image.open(ROOT / r.split / 'images' / r.file)); ax.set_title(f'{lab} {r.decoration_share:.2f}', fontsize=9)
    for ax in row:
        ax.axis('off')
plt.tight_layout(); plt.show()

## 2. Baseline: ufal model unchanged, on the DIVA test split

Same mapping as for our own pages: `TEXT_HW` / `TEXT` → codex, `DRAW` / `PHOTO` → illuminated.

In [ ]:
import torch
from transformers import AutoImageProcessor, AutoModelForImageClassification

MODEL, REV = 'ufal/vit-historical-page', 'v4.4'
UFAL = ['DRAW', 'DRAW_L', 'LINE_HW', 'LINE_P', 'LINE_T', 'PHOTO',
        'PHOTO_L', 'TEXT', 'TEXT_HW', 'TEXT_P', 'TEXT_T']
TO_OURS = {'TEXT_HW': 'codex', 'TEXT': 'codex', 'DRAW': 'illuminated', 'DRAW_L': 'illuminated',
           'PHOTO': 'illuminated', 'PHOTO_L': 'illuminated'}
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
proc = AutoImageProcessor.from_pretrained(MODEL, revision=REV)
base = AutoModelForImageClassification.from_pretrained(MODEL, revision=REV).eval().to(dev)

@torch.no_grad()
def predict(model, paths, batch=16):
    out = []
    for i in range(0, len(paths), batch):
        imgs = [Image.open(p).convert('RGB') for p in paths[i:i + batch]]
        logits = model(**proc(images=imgs, return_tensors='pt').to(dev)).logits
        out += logits.argmax(-1).tolist()
    return out

test = diva[(diva.split == 'test') & (diva.label != 'unsure')].copy()
test['ufal'] = [UFAL[i] for i in predict(base, [ROOT / 'test' / 'images' / f for f in test.file])]
test['ufal_mapped'] = test['ufal'].map(TO_OURS).fillna('other')
print('ufal labels per weak label:'); display(pd.crosstab(test['label'], test['ufal']))
print(f"Baseline accuracy (codex vs illuminated): {(test['ufal_mapped'] == test['label']).mean():.1%} on {len(test)} pages")

## 3. First fine-tune: codex vs illuminated

The ufal model gets a new last layer with 2 classes and is trained on the DIVA train split
(pages labelled `unsure` are left out), checked on the val split after each epoch.
Takes a few minutes on a T4. If you get *CUDA out of memory*, set `BATCH = 8`.

**Caveat:** train, val and test pages come from the **same six books**, so the DIVA test score
is optimistic. Step 4 is the honest check.

In [ ]:
import random
from torch.utils.data import DataLoader, Dataset

CLASSES = ['codex', 'illuminated']
EPOCHS, BATCH, LR = 3, 16, 5e-5
torch.manual_seed(0); random.seed(0)

class Pages(Dataset):
    def __init__(self, df, train=False):
        self.items = [(ROOT / r.split / 'images' / r.file, CLASSES.index(r.label)) for r in df.itertuples()]
        self.train = train
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, y = self.items[i]
        img = Image.open(path).convert('RGB')
        if self.train:   # light augmentation that keeps the layout: small brightness change only
            img = Image.eval(img, lambda v, f=random.uniform(0.85, 1.15): min(255, int(v * f)))
        return img, y

def collate(batch):
    imgs, ys = zip(*batch)
    x = proc(images=list(imgs), return_tensors='pt')['pixel_values']
    return x, torch.tensor(ys)

usable = diva[diva.label != 'unsure']
train_dl = DataLoader(Pages(usable[usable.split == 'train'], train=True), batch_size=BATCH, shuffle=True, collate_fn=collate, num_workers=2)
val_dl = DataLoader(Pages(usable[usable.split == 'val']), batch_size=BATCH, collate_fn=collate, num_workers=2)

model = AutoModelForImageClassification.from_pretrained(
    MODEL, revision=REV, num_labels=len(CLASSES), ignore_mismatched_sizes=True,
    id2label=dict(enumerate(CLASSES)), label2id={c: i for i, c in enumerate(CLASSES)}).to(dev)

# the classes are unbalanced, so weight the loss by 1 / class frequency
counts = usable[usable.split == 'train'].label.value_counts().reindex(CLASSES).values
loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor(counts.sum() / (len(CLASSES) * counts), dtype=torch.float).to(dev))
opt = torch.optim.AdamW(model.parameters(), lr=LR)

def accuracy(dl):
    model.eval(); right = n = 0
    with torch.no_grad():
        for x, y in dl:
            right += (model(pixel_values=x.to(dev)).logits.argmax(-1).cpu() == y).sum().item(); n += len(y)
    return right / n

for epoch in range(EPOCHS):
    model.train(); total = 0
    for x, y in train_dl:
        opt.zero_grad()
        loss = loss_fn(model(pixel_values=x.to(dev)).logits, y.to(dev))
        loss.backward(); opt.step(); total += loss.item() * len(y)
    print(f'epoch {epoch + 1}: train loss {total / len(train_dl.dataset):.3f}, val accuracy {accuracy(val_dl):.1%}')

test_dl = DataLoader(Pages(usable[usable.split == 'test']), batch_size=BATCH, collate_fn=collate)
print(f'DIVA test accuracy after fine-tuning: {accuracy(test_dl):.1%} (baseline above)')
model.save_pretrained('/content/drive/MyDrive/doctype-codex-illuminated-v0')
proc.save_pretrained('/content/drive/MyDrive/doctype-codex-illuminated-v0')

## 4. The real check: our own codex and illuminated test pages

These pages come from other sources than DIVA (different books, scans and crops).
If the model still separates codex from illuminated here, it learned the layout, not the collection.

In [ ]:
ours = pd.read_csv(PAGES / 'labels.csv')
ours = ours[ours.type.isin(CLASSES)].copy()
model.eval()
ours['finetuned'] = [CLASSES[i] for i in predict(model, [PAGES / f for f in ours.file])]
ours['ufal'] = [TO_OURS.get(UFAL[i], 'other') for i in predict(base, [PAGES / f for f in ours.file])]
print(f"Our pages ({len(ours)}): baseline {(ours.ufal == ours.type).mean():.1%}, "
      f"fine-tuned {(ours.finetuned == ours.type).mean():.1%}")
display(pd.crosstab(ours['type'], ours['finetuned'], rownames=['our label'], colnames=['fine-tuned model']))
display(ours[ours.finetuned != ours.type][['file', 'type', 'finetuned', 'ufal']])
ours.to_csv('/content/drive/MyDrive/diva_finetune_on_our_pages.csv', index=False)

## What this does and does not show

- It is a first, two-class model. Charter, register and other are not in DIVA; they need data
  from other collections (Königsfelden charters, e-NDP registers, printed books).
- The weak labels come from a threshold on the masks, not from people. Check the sample grid.
- DIVA is one kind of book (richly decorated books of hours), so an "illuminated" here is
  a page with painted borders; our own illuminated pages are more varied.